# Retrieval-Augmented Generation (RAG)

**Retrieval-Augmented Generation (RAG)** is an architecture that combines:

1. **Information Retrieval** — finding relevant information from a knowledge base.
2. **Large Language Models (LLMs)** — generating a natural-language answer using the retrieved information.

Instead of asking an LLM to answer entirely from its pretrained knowledge, RAG first retrieves relevant information from an external knowledge base and provides that information to the LLM as context.


### LLMs have several limitations:

- Their knowledge may not contain private or domain-specific documents.
- Their knowledge can become outdated.
- They can hallucinate information.
- They cannot automatically access an organization's internal documents.
- They may not provide answers grounded in the exact source material.

RAG addresses these problems by connecting the LLM to an external knowledge base.

### Traditional LLM

```text
User Question
      ↓
     LLM
      ↓
   Answer
```

The LLM relies primarily on its pretrained knowledge.

### RAG

```text
User Question
      ↓
    Retriever
      ↓
Knowledge Base
      ↓
Relevant Documents
      ↓
     LLM
      ↓
   Answer
```

The LLM receives external context before generating the answer.




In [5]:
# =============================================================================
# IMPORT LIBRARIES
# =============================================================================

from __future__ import annotations

from pathlib import Path
import re
from typing import Optional

import numpy as np
import pymupdf
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from rapidfuzz import fuzz

In [4]:
# =============================================================================
# CONFIGURATION
# =============================================================================

BASE_DIR = Path.cwd().parent
PROJECT_DIR = BASE_DIR.resolve().parents[1]
DATA_DIR = BASE_DIR / "data"

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

DEFAULT_TOP_K = 5
DEFAULT_CANDIDATE_K = 50

# RRF constant
RRF_K = 60

In [ ]:
# =============================================================================
# DOCUMENT DISCOVERY
# =============================================================================

def discover_pdfs(data_dir: Path = DATA_DIR) -> list[Path]:
    """
    Recursively discover all PDF files.
    """

    return sorted(data_dir.rglob("*.pdf"))

In [ ]:
# =============================================================================
# PDF EXTRACTION
# =============================================================================

def extract_pdf_pages(pdf_path: Path) -> list[dict]:
    """
    Extract page-level text from a PDF using PyMuPDF.
    """

    pages = []

    with pymupdf.open(pdf_path) as document:

        for page_number, page in enumerate(document, start=1):

            text = page.get_text("text")

            pages.append({
                "page_number": page_number,
                "text": text,
            })

    return pages

In [ ]:
# =============================================================================
# NORMALIZATION
# =============================================================================

def normalize_section_number(
    value: Optional[str],
) -> Optional[str]:
    """
    Normalize section numbers.

    Examples:

        "420"   -> "420"
        "420A"  -> "420A"
        " 420 " -> "420"
    """

    if not value:
        return None

    value = value.strip().upper()

    value = value.rstrip(".")

    return value

In [ ]:
def normalize_document_name(document: str) -> str:
    """
    Normalize document filenames into predictable document IDs.
    """

    value = document.lower().strip()

    # Remove common file extensions
    value = re.sub(r"\.(pdf|txt)$", "", value)

    aliases = {

        # IPC
        "ipc": "ipc_1860",
        "ipc1860": "ipc_1860",
        "ipc_1860": "ipc_1860",
        "indian_penal_code": "ipc_1860",

        # CrPC
        "crpc": "crpc_1973",
        "crpc1973": "crpc_1973",
        "crpc_1973": "crpc_1973",

        # BNSS
        "bnss": "bnss_2023",
        "bnss2023": "bnss_2023",
        "bnss_2023": "bnss_2023",

        # BNS
        "bns": "bns_2023",
        "bns2023": "bns_2023",
        "bns_2023": "bns_2023",

        # Contract Act
        "contract": "contract_act",
        "contract_act": "contract_act",
        "indian_contract_act": "contract_act",

        # Evidence Act
        "evidence": "evidence_act_1872",
        "evidence_act": "evidence_act_1872",
        "evidence_act_1872": "evidence_act_1872",
        "indian_evidence_act": "evidence_act_1872",
    }

    return aliases.get(value, value)

In [ ]:
# =============================================================================
# ACT DISPLAY NAMES
# =============================================================================

ACT_NAMES = {
    "ipc_1860": "Indian Penal Code, 1860",
    "crpc_1973": "Code of Criminal Procedure, 1973",
    "bnss_2023": "Bharatiya Nagarik Suraksha Sanhita, 2023",
    "bns_2023": "Bharatiya Nyaya Sanhita, 2023",
    "contract_act": "Indian Contract Act, 1872",
    "evidence_act_1872": "Indian Evidence Act, 1872",
}


In [ ]:
# =============================================================================
# TOKENIZATION
# =============================================================================

def tokenize(text: str) -> list[str]:
    """
    BM25 tokenization.

    Keeps:
        - words
        - numbers
        - alphanumeric legal identifiers
    """

    return re.findall(
        r"\b[a-zA-Z0-9]+\b",
        text.lower(),
    )

In [ ]:
# =============================================================================
# FUZZY MATCHING
# =============================================================================

def fuzzy_contains(
    query: str,
    terms: list[str],
    threshold: int = 82,
) -> bool:
    """
    Detect misspelled words using RapidFuzz.

    Example:

        fraus -> fraud
        fruad -> fraud
        cheet -> cheat
        deceve -> deceive
    """

    words = re.findall(
        r"\b[a-zA-Z]+\b",
        query.lower(),
    )

    for word in words:

        for term in terms:

            # Exact
            if word == term:
                return True

            # Fuzzy
            score = fuzz.ratio(word, term)

            if score >= threshold:
                return True

    return False

In [ ]:
def fuzzy_word_matches(
    query: str,
    terms: list[str],
    threshold: int = 82,
) -> list[str]:
    """
    Return legal terms that approximately match words in the query.
    """

    words = re.findall(
        r"\b[a-zA-Z]+\b",
        query.lower(),
    )

    matches = []

    for word in words:

        for term in terms:

            if word == term:
                matches.append(term)
                continue

            score = fuzz.ratio(word, term)

            if score >= threshold:
                matches.append(term)

    return list(dict.fromkeys(matches))

In [ ]:
# =============================================================================
# SECTION EXTRACTION
# =============================================================================

def extract_section_number(
    query: str,
) -> Optional[str]:
    """
    Extract legal section number.

    Examples:

        what is 420?
        what is section 420?
        explain section 420 IPC
        IPC 420
        section 420A
    """

    patterns = [

        # Section 420
        r"\bsection\s+(\d+[A-Z]?)\b",

        # Sec 420
        r"\bsec\.?\s+(\d+[A-Z]?)\b",

        # IPC 420 / BNS 420
        r"\b(?:ipc|crpc|bnss|bns)\s+(\d+[A-Z]?)\b",

        # Standalone number
        r"\b(\d+[A-Z]?)\b",
    ]

    for pattern in patterns:

        match = re.search(
            pattern,
            query,
            re.IGNORECASE,
        )

        if match:

            return normalize_section_number(
                match.group(1)
            )

    return None


In [ ]:
# =============================================================================
# ACT DETECTION
# =============================================================================

def detect_act(query: str) -> Optional[str]:
    """
    Detect the Act mentioned in a query.
    """

    query_lower = query.lower()

    patterns = {

        "ipc_1860": [
            r"\bipc\b",
            r"indian penal code",
            r"penal code",
        ],

        "crpc_1973": [
            r"\bcrpc\b",
            r"criminal procedure code",
            r"code of criminal procedure",
        ],

        "bnss_2023": [
            r"\bbnss\b",
            r"bharatiya nagarik suraksha sanhita",
        ],

        "bns_2023": [
            r"\bbns\b",
            r"bharatiya nyaya sanhita",
        ],

        "contract_act": [
            r"\bcontract act\b",
            r"indian contract act",
        ],

        "evidence_act_1872": [
            r"\bevidence act\b",
            r"indian evidence act",
        ],
    }

    for act, act_patterns in patterns.items():

        for pattern in act_patterns:

            if re.search(
                pattern,
                query_lower,
            ):
                return act

    return None


In [ ]:
# =============================================================================
# INTENT DETECTION
# =============================================================================

def detect_intent(query: str) -> str:
    """
    Detect broad legal intent.
    """

    query_lower = query.lower()

    # Definition
    if re.search(
        r"\b("
        r"what is|"
        r"what does|"
        r"explain|"
        r"meaning of|"
        r"define|"
        r"tell me about"
        r")\b",
        query_lower,
    ):
        return "definition"

    # Legal action
    if re.search(
        r"\b("
        r"what case|"
        r"which case|"
        r"can i file|"
        r"can i take|"
        r"what complaint|"
        r"legal action|"
        r"what action|"
        r"what offence|"
        r"which offence|"
        r"what section|"
        r"which section|"
        r"what law|"
        r"which law|"
        r"what can i do|"
        r"how can i proceed"
        r")\b",
        query_lower,
    ):
        return "legal_action"

    # Punishment
    if re.search(
        r"\b("
        r"punishment|"
        r"sentence|"
        r"jail|"
        r"imprisonment|"
        r"fine|"
        r"penalty|"
        r"punished|"
        r"punishable"
        r")\b",
        query_lower,
    ):
        return "punishment"

    # Evidence
    if re.search(
        r"\b("
        r"evidence|"
        r"prove|"
        r"proof|"
        r"documents|"
        r"document|"
        r"admissible|"
        r"admissibility|"
        r"witness"
        r")\b",
        query_lower,
    ):
        return "evidence"

    # Procedure
    if re.search(
        r"\b("
        r"procedure|"
        r"process|"
        r"how to file|"
        r"how do i file|"
        r"complaint|"
        r"cognizance|"
        r"investigation|"
        r"arrest|"
        r"trial"
        r")\b",
        query_lower,
    ):
        return "procedure"

    # Comparison
    if re.search(
        r"\b("
        r"difference|"
        r"compare|"
        r"comparison|"
        r"equivalent|"
        r"corresponding|"
        r"replacement|"
        r"replaced by"
        r")\b",
        query_lower,
    ):
        return "comparison"

    return "general"


In [ ]:
# =============================================================================
# LEGAL CONCEPT DETECTION
# =============================================================================

def detect_legal_concepts(
    query: str,
) -> tuple[list[str], list[str]]:
    """
    Detect legal concepts and fuzzy-matched terms.

    Returns:

        concepts
        fuzzy_matches
    """

    query_lower = query.lower()

    concepts = []

    # -------------------------------------------------------------------------
    # FRAUD / CHEATING
    # -------------------------------------------------------------------------

    fraud_terms = [
        "fraud",
        "fraudulent",
        "cheat",
        "cheating",
        "deceive",
        "deception",
        "dishonest",
        "dishonestly",
        "inducement",
        "induce",
        "false representation",
        "false promise",
        "fraudulently",
        "dishonest inducement",
    ]

    fraud_typos = [
        "fraus",
        "fruad",
        "fraude",
        "fraudl",
        "fraudd",
        "frausd",
        "cheet",
        "cheetd",
        "cheetng",
        "cheatingg",
        "deceve",
        "deceved",
        "deceivee",
        "dishonst",
        "dishonestlyy",
        "inducemnt",
    ]

    fraud_detected = False

    if any(
        term in query_lower
        for term in fraud_terms
    ):
        fraud_detected = True

    if any(
        typo in query_lower
        for typo in fraud_typos
    ):
        fraud_detected = True

    if fuzzy_contains(
        query,
        fraud_terms,
        threshold=82,
    ):
        fraud_detected = True

    if fraud_detected:

        concepts.extend([
            "fraud",
            "fraudulent",
            "cheating",
            "cheat",
            "deception",
            "deceive",
            "dishonest",
            "dishonestly",
            "dishonest inducement",
            "dishonestly inducing",
            "inducement",
            "false representation",
            "false promise",
        ])

    # -------------------------------------------------------------------------
    # PROPERTY / MONEY
    # -------------------------------------------------------------------------

    property_terms = [
        "money",
        "property",
        "amount",
        "payment",
        "cash",
        "bank",
        "transfer",
        "fund",
        "funds",
        "valuable security",
        "delivery of property",
    ]

    if any(
        term in query_lower
        for term in property_terms
    ):

        concepts.extend([
            "property",
            "money",
            "payment",
            "dishonest inducement",
            "delivery of property",
            "valuable security",
        ])

    # -------------------------------------------------------------------------
    # CONTRACT
    # -------------------------------------------------------------------------

    contract_terms = [
        "contract",
        "agreement",
        "breach",
        "promise",
        "payment",
        "consideration",
        "consent",
        "free consent",
        "void contract",
        "misrepresentation",
        "coercion",
        "undue influence",
    ]

    if any(
        term in query_lower
        for term in contract_terms
    ):

        concepts.extend([
            "contract",
            "agreement",
            "breach",
            "promise",
            "consideration",
            "consent",
            "free consent",
            "misrepresentation",
            "coercion",
            "undue influence",
        ])

    # -------------------------------------------------------------------------
    # FORGERY
    # -------------------------------------------------------------------------

    forgery_terms = [
        "forgery",
        "forge",
        "forged",
        "fake document",
        "false document",
        "fabricated document",
        "fabrication",
    ]

    if fuzzy_contains(
        query,
        forgery_terms,
        threshold=82,
    ):

        concepts.extend([
            "forgery",
            "forged",
            "false document",
            "fabricated document",
        ])

    # -------------------------------------------------------------------------
    # THREAT / INTIMIDATION
    # -------------------------------------------------------------------------

    threat_terms = [
        "threat",
        "threaten",
        "threatened",
        "intimidation",
        "intimidate",
        "criminal intimidation",
    ]

    if fuzzy_contains(
        query,
        threat_terms,
        threshold=82,
    ):

        concepts.extend([
            "threat",
            "criminal intimidation",
            "intimidation",
        ])

    # -------------------------------------------------------------------------
    # ASSAULT / HURT
    # -------------------------------------------------------------------------

    assault_terms = [
        "assault",
        "attack",
        "beaten",
        "beat me",
        "hit me",
        "hurt",
        "injury",
        "injured",
        "physical attack",
    ]

    if fuzzy_contains(
        query,
        assault_terms,
        threshold=82,
    ):

        concepts.extend([
            "assault",
            "hurt",
            "injury",
            "physical attack",
        ])

    # -------------------------------------------------------------------------
    # PROPERTY DISPUTE
    # -------------------------------------------------------------------------

    property_dispute_terms = [
        "land",
        "property dispute",
        "house",
        "ownership",
        "possession",
        "encroachment",
        "boundary",
        "tenant",
        "rent",
    ]

    if any(
        term in query_lower
        for term in property_dispute_terms
    ):

        concepts.extend([
            "property",
            "ownership",
            "possession",
            "encroachment",
            "property dispute",
        ])

    # -------------------------------------------------------------------------
    # EVIDENCE
    # -------------------------------------------------------------------------

    evidence_terms = [
        "evidence",
        "proof",
        "prove",
        "document",
        "documents",
        "witness",
        "statement",
        "confession",
        "admissible",
    ]

    if any(
        term in query_lower
        for term in evidence_terms
    ):

        concepts.extend([
            "evidence",
            "proof",
            "witness",
            "statement",
            "confession",
            "admissible",
        ])

    # Remove duplicates
    concepts = list(
        dict.fromkeys(concepts)
    )

    fuzzy_matches = fuzzy_word_matches(
        query,
        fraud_terms
        + property_terms
        + contract_terms
        + forgery_terms
        + threat_terms
        + assault_terms,
        threshold=82,
    )

    return concepts, fuzzy_matches

In [ ]:



















# =============================================================================
# QUERY ANALYSIS
# =============================================================================

def analyze_query(query: str) -> dict:
    """
    Convert raw legal query into structured information.
    """

    section_number = extract_section_number(
        query
    )

    act = detect_act(query)

    intent = detect_intent(query)

    concepts, fuzzy_matches = detect_legal_concepts(
        query
    )

    return {
        "original_query": query,
        "section_number": section_number,
        "act": act,
        "intent": intent,
        "concepts": concepts,
        "fuzzy_matches": fuzzy_matches,
        "is_section_query": section_number is not None,
    }


# =============================================================================
# LEGAL DOCUMENT PARSER
# =============================================================================

def parse_legal_document(
    pages: list[dict],
    document: str,
    category: str,
) -> list[dict]:
    """
    Parse a legal PDF into section-level chunks.

    Each chunk contains:

        text
        page_number
        chapter
        section
        section_number
        section_title
        subsection
        document
        document_id
        category
    """

    chunks = []

    current_chapter = None
    current_section = None
    current_section_number = None
    current_section_title = None
    current_subsection = None

    current_text = []
    current_page = None

    document_id = normalize_document_name(
        document
    )

    def save_chunk():

        nonlocal current_text

        if not current_text:
            return

        text = "\n".join(
            current_text
        ).strip()

        if not text:
            return

        chunks.append({

            "text": text,

            "page_number": current_page,

            "section": current_section,

            "section_number": current_section_number,

            "section_title": current_section_title,

            "subsection": current_subsection,

            "chapter": current_chapter,

            "document": document,

            "document_id": document_id,

            "category": category,
        })

    for page in pages:

        page_number = page["page_number"]

        lines = page["text"].splitlines()

        for raw_line in lines:

            line = re.sub(
                r"\s+",
                " ",
                raw_line,
            ).strip()

            if not line:
                continue

            # =================================================================
            # CHAPTER
            # =================================================================

            chapter_match = re.match(
                r"^(CHAPTER\s+[IVXLCDM0-9]+.*)$",
                line,
                re.IGNORECASE,
            )

            if chapter_match:

                current_chapter = (
                    chapter_match.group(1).strip()
                )

                continue

            # =================================================================
            # SECTION WITH PERIOD
            # =================================================================

            section_match = re.match(
                r"^(?:SECTION\s+)?(\d+[A-Z]?)\.\s*(.*)$",
                line,
                re.IGNORECASE,
            )

            if section_match:

                save_chunk()

                current_section_number = (
                    normalize_section_number(
                        section_match.group(1)
                    )
                )

                current_section = (
                    f"Section "
                    f"{current_section_number}"
                )

                current_section_title = None

                current_subsection = None

                current_text = [line]

                current_page = page_number

                # The text after "420." is often the section title.
                remainder = (
                    section_match.group(2).strip()
                )

                if remainder:

                    current_section_title = (
                        remainder
                    )

                continue

            # =================================================================
            # SECTION WITHOUT PERIOD
            # =================================================================

            section_no_period = re.match(
                r"^SECTION\s+(\d+[A-Z]?)\s*$",
                line,
                re.IGNORECASE,
            )

            if section_no_period:

                save_chunk()

                current_section_number = (
                    normalize_section_number(
                        section_no_period.group(1)
                    )
                )

                current_section = (
                    f"Section "
                    f"{current_section_number}"
                )

                current_section_title = None

                current_subsection = None

                current_text = [line]

                current_page = page_number

                continue

            # =================================================================
            # SUBSECTION
            # =================================================================

            subsection_match = re.match(
                r"^(\([0-9A-Za-z]+\))\s*(.*)$",
                line,
            )

            if (
                subsection_match
                and current_section
            ):

                current_subsection = (
                    subsection_match.group(1)
                )

                current_text.append(
                    f"{current_subsection} "
                    f"{subsection_match.group(2)}".strip()
                )

                continue

            # =================================================================
            # NORMAL TEXT
            # =================================================================

            if current_text:

                current_text.append(line)

            else:

                current_text = [line]

                current_page = page_number

    save_chunk()

    return chunks


# =============================================================================
# PROCESS DOCUMENTS
# =============================================================================

def process_documents(
    pdf_files: list[Path],
) -> list[dict]:

    all_chunks = []

    for pdf_path in pdf_files:

        try:

            pages = extract_pdf_pages(
                pdf_path
            )

            chunks = parse_legal_document(
                pages=pages,
                document=pdf_path.stem,
                category=pdf_path.parent.name,
            )

            all_chunks.extend(
                chunks
            )

            print(
                f"{pdf_path.name}: "
                f"{len(pages)} pages -> "
                f"{len(chunks)} chunks"
            )

        except Exception as error:

            print(
                f"ERROR: "
                f"{pdf_path}: "
                f"{error}"
            )

    return all_chunks


# =============================================================================
# LEGAL CONCEPT SCORE
# =============================================================================

def legal_concept_score(
    chunk: dict,
    concepts: list[str],
) -> float:
    """
    Score how strongly a chunk matches legal concepts.

    Section title matches receive much more weight than body matches.
    """

    if not concepts:
        return 0.0

    title = (
        chunk.get("section_title")
        or ""
    ).lower()

    text = (
        chunk.get("text")
        or ""
    ).lower()

    score = 0.0

    for concept in concepts:

        concept_lower = concept.lower()

        # -------------------------------------------------------------
        # Section title
        # -------------------------------------------------------------

        if concept_lower in title:

            score += 0.40

        # -------------------------------------------------------------
        # Body
        # -------------------------------------------------------------

        if concept_lower in text:

            score += 0.15

    return min(
        score,
        1.50,
    )


# =============================================================================
# INTENT SCORE
# =============================================================================

def legal_intent_score(
    chunk: dict,
    intent: str,
) -> float:

    text = (
        chunk.get("text")
        or ""
    ).lower()

    title = (
        chunk.get("section_title")
        or ""
    ).lower()

    combined = (
        title
        + " "
        + text
    )

    score = 0.0

    if intent == "punishment":

        terms = [
            "punished",
            "punishment",
            "imprisonment",
            "fine",
            "penalty",
            "punishable",
        ]

        if any(
            term in combined
            for term in terms
        ):
            score += 0.25

    elif intent == "evidence":

        terms = [
            "evidence",
            "proof",
            "witness",
            "statement",
            "confession",
            "admissible",
        ]

        if any(
            term in combined
            for term in terms
        ):
            score += 0.25

    elif intent == "procedure":

        terms = [
            "complaint",
            "procedure",
            "cognizance",
            "investigation",
            "arrest",
            "trial",
            "magistrate",
            "court",
        ]

        if any(
            term in combined
            for term in terms
        ):
            score += 0.20

    elif intent == "legal_action":

        terms = [
            "offence",
            "offense",
            "complaint",
            "punished",
            "cheating",
            "fraud",
            "dishonest",
            "deception",
            "inducement",
        ]

        if any(
            term in combined
            for term in terms
        ):
            score += 0.20

    elif intent == "definition":

        # Definitions generally benefit from title/body overlap.
        if title:
            score += 0.05

    return score


# =============================================================================
# LEGAL RELEVANCE PENALTY
# =============================================================================

def legal_relevance_penalty(
    chunk: dict,
    query_info: dict,
) -> float:
    """
    Penalize obviously irrelevant documents.

    Important:

    This does NOT completely remove candidates.

    It only lowers their ranking.
    """

    concepts = query_info.get(
        "concepts",
        [],
    )

    intent = query_info.get(
        "intent"
    )

    if not concepts:
        return 0.0

    text = (
        chunk.get("text")
        or ""
    ).lower()

    title = (
        chunk.get("section_title")
        or ""
    ).lower()

    combined = (
        title
        + " "
        + text
    )

    # -------------------------------------------------------------------------
    # Fraud query
    # -------------------------------------------------------------------------

    fraud_concepts = {
        "fraud",
        "fraudulent",
        "cheating",
        "cheat",
        "deception",
        "deceive",
        "dishonest",
        "dishonestly",
        "dishonest inducement",
        "dishonestly inducing",
        "inducement",
        "false representation",
        "false promise",
    }

    query_is_fraud = bool(
        fraud_concepts.intersection(
            set(concepts)
        )
    )

    if query_is_fraud:

        fraud_terms = [
            "fraud",
            "fraudulent",
            "cheat",
            "cheating",
            "deceive",
            "deception",
            "dishonest",
            "dishonestly",
            "inducement",
            "false representation",
            "false promise",
        ]

        has_fraud_context = any(
            term in combined
            for term in fraud_terms
        )

        if not has_fraud_context:

            return -0.35

    # -------------------------------------------------------------------------
    # Legal action query
    # -------------------------------------------------------------------------

    if intent == "legal_action":

        legal_terms = [
            "offence",
            "offense",
            "punished",
            "punishable",
            "complaint",
            "cheating",
            "fraud",
            "dishonest",
            "deception",
            "inducement",
            "court",
            "magistrate",
        ]

        has_legal_context = any(
            term in combined
            for term in legal_terms
        )

        if not has_legal_context:

            return -0.15

    return 0.0


# =============================================================================
# HYBRID RETRIEVER
# =============================================================================

class HybridRetriever:

    """
    Legal-aware hybrid retriever.

    Retrieval layers:

        1. BM25
        2. Dense embeddings
        3. RRF
        4. Exact section
        5. Exact Act + section
        6. Legal concept scoring
        7. Intent scoring
        8. Act-aware ranking
        9. Irrelevance penalty
        10. Deduplication
    """

    def __init__(
        self,
        chunks: list[dict],
        model_name: str = EMBEDDING_MODEL,
    ):

        self.chunks = [
            chunk
            for chunk in chunks
            if chunk.get(
                "text",
                "",
            ).strip()
        ]

        if not self.chunks:

            raise ValueError(
                "No non-empty chunks available."
            )

        self.texts = [
            chunk["text"]
            for chunk in self.chunks
        ]

        # =====================================================================
        # BM25
        # =====================================================================

        print(
            "Building BM25 index..."
        )

        tokenized_texts = [
            tokenize(text)
            for text in self.texts
        ]

        self.bm25 = BM25Okapi(
            tokenized_texts
        )

        # =====================================================================
        # EMBEDDINGS
        # =====================================================================

        print(
            f"Loading embedding model: "
            f"{model_name}"
        )

        self.embedding_model = SentenceTransformer(
            model_name
        )

        print(
            "Creating embeddings..."
        )

        self.embeddings = (
            self.embedding_model.encode(
                self.texts,
                normalize_embeddings=True,
                show_progress_bar=True,
            )
        )

        # =====================================================================
        # EXACT SECTION INDEX
        # =====================================================================

        self.section_index: dict[
            str,
            list[int],
        ] = {}

        # =====================================================================
        # ACT + SECTION INDEX
        # =====================================================================

        self.act_section_index: dict[
            tuple[str, str],
            list[int],
        ] = {}

        for index, chunk in enumerate(
            self.chunks
        ):

            section_number = normalize_section_number(
                chunk.get(
                    "section_number"
                )
            )

            document_id = normalize_document_name(
                chunk.get(
                    "document_id",
                    "",
                )
            )

            if not section_number:
                continue

            # -------------------------------------------------------------
            # Section index
            # -------------------------------------------------------------

            self.section_index.setdefault(
                section_number,
                [],
            ).append(index)

            # -------------------------------------------------------------
            # Act + section index
            # -------------------------------------------------------------

            if document_id:

                key = (
                    document_id,
                    section_number,
                )

                self.act_section_index.setdefault(
                    key,
                    [],
                ).append(index)

        print(
            f"Indexed "
            f"{len(self.section_index)} "
            f"unique section numbers."
        )

        print(
            f"Indexed "
            f"{len(self.act_section_index)} "
            f"Act + section combinations."
        )

    # =========================================================================
    # EXACT SECTION SEARCH
    # =========================================================================

    def exact_section_search(
        self,
        section_number: str,
        act: Optional[str] = None,
    ) -> list[int]:

        section_number = (
            normalize_section_number(
                section_number
            )
        )

        if not section_number:
            return []

        # Exact Act + section
        if act:

            key = (
                normalize_document_name(act),
                section_number,
            )

            return self.act_section_index.get(
                key,
                [],
            )

        # Exact section across all Acts
        return self.section_index.get(
            section_number,
            [],
        )

    # =========================================================================
    # ACT MATCH SCORE
    # =========================================================================

    def act_match_score(
        self,
        chunk: dict,
        act: Optional[str],
        section_number: Optional[str],
    ) -> float:

        if not act:
            return 0.0

        chunk_act = normalize_document_name(
            chunk.get(
                "document_id",
                "",
            )
        )

        if chunk_act != act:
            return 0.0

        # Explicit Act mentioned
        if section_number:

            chunk_section = normalize_section_number(
                chunk.get(
                    "section_number"
                )
            )

            if chunk_section == section_number:

                # Very strong boost
                return 2.50

            # Correct Act but another section
            return 0.25

        # Act mentioned but no section
        return 0.50

    # =========================================================================
    # SECTION TITLE SCORE
    # =========================================================================

    def section_title_score(
        self,
        chunk: dict,
        concepts: list[str],
    ) -> float:

        if not concepts:
            return 0.0

        title = (
            chunk.get(
                "section_title"
            )
            or ""
        ).lower()

        if not title:
            return 0.0

        score = 0.0

        for concept in concepts:

            if concept.lower() in title:

                score += 0.35

        return min(
            score,
            1.0,
        )

    # =========================================================================
    # SEARCH
    # =========================================================================

    def search(
        self,
        query: str,
        top_k: int = DEFAULT_TOP_K,
        candidate_k: int = DEFAULT_CANDIDATE_K,
    ) -> list[dict]:

        query_info = analyze_query(
            query
        )

        section_number = query_info[
            "section_number"
        ]

        act = query_info[
            "act"
        ]

        intent = query_info[
            "intent"
        ]

        concepts = query_info[
            "concepts"
        ]

        # =====================================================================
        # QUERY EXPANSION
        # =====================================================================

        expanded_query = query

        if concepts:

            expanded_query += " "

            expanded_query += " ".join(
                concepts
            )

        # =====================================================================
        # BM25
        # =====================================================================

        query_tokens = tokenize(
            expanded_query
        )

        sparse_scores = (
            self.bm25.get_scores(
                query_tokens
            )
        )

        sparse_indices = np.argsort(
            sparse_scores
        )[::-1][
            :candidate_k
        ]

        # =====================================================================
        # DENSE
        # =====================================================================

        dense_query = expanded_query

        query_embedding = (
            self.embedding_model.encode(
                [dense_query],
                normalize_embeddings=True,
            )
        )

        dense_scores = (
            cosine_similarity(
                query_embedding,
                self.embeddings,
            )[0]
        )

        dense_indices = np.argsort(
            dense_scores
        )[::-1][
            :candidate_k
        ]

        # =====================================================================
        # RRF FUSION
        # =====================================================================

        fused_scores: dict[
            int,
            float,
        ] = {}

        for rank, index in enumerate(
            sparse_indices,
            start=1,
        ):

            index = int(index)

            fused_scores[index] = (
                fused_scores.get(
                    index,
                    0.0,
                )
                + 1.0
                / (
                    RRF_K
                    + rank
                )
            )

        for rank, index in enumerate(
            dense_indices,
            start=1,
        ):

            index = int(index)

            fused_scores[index] = (
                fused_scores.get(
                    index,
                    0.0,
                )
                + 1.0
                / (
                    RRF_K
                    + rank
                )
            )

        # =====================================================================
        # EXACT SECTION CANDIDATES
        # =====================================================================

        exact_indices = []

        if section_number:

            exact_indices = (
                self.exact_section_search(
                    section_number=section_number,
                    act=act,
                )
            )

            for index in exact_indices:

                if index not in fused_scores:

                    fused_scores[index] = 0.0

        # =====================================================================
        # FINAL SCORING
        # =====================================================================

        final_scores = {}

        diagnostics = {}

        for index, rrf_score in fused_scores.items():

            chunk = self.chunks[index]

            score = float(
                rrf_score
            )

            chunk_section = (
                normalize_section_number(
                    chunk.get(
                        "section_number"
                    )
                )
            )

            chunk_act = (
                normalize_document_name(
                    chunk.get(
                        "document_id",
                        "",
                    )
                )
            )

            # -------------------------------------------------------------
            # Exact section
            # -------------------------------------------------------------

            exact_section = (
                section_number is not None
                and chunk_section == section_number
            )

            exact_act = (
                act is not None
                and chunk_act == act
            )

            exact_act_section = (
                exact_section
                and exact_act
            )

            if exact_section:

                score += 1.00

            # -------------------------------------------------------------
            # Exact Act + Section
            # -------------------------------------------------------------

            if exact_act_section:

                score += 2.50

            # -------------------------------------------------------------
            # Explicit Act match
            # -------------------------------------------------------------

            act_score = (
                self.act_match_score(
                    chunk=chunk,
                    act=act,
                    section_number=section_number,
                )
            )

            score += act_score

            # -------------------------------------------------------------
            # Concept score
            # -------------------------------------------------------------

            concept_score = (
                legal_concept_score(
                    chunk,
                    concepts,
                )
            )

            score += concept_score

            # -------------------------------------------------------------
            # Section title score
            # -------------------------------------------------------------

            title_score = (
                self.section_title_score(
                    chunk,
                    concepts,
                )
            )

            score += title_score

            # -------------------------------------------------------------
            # Intent score
            # -------------------------------------------------------------

            intent_score = (
                legal_intent_score(
                    chunk,
                    intent,
                )
            )

            score += intent_score

            # -------------------------------------------------------------
            # Relevance penalty
            # -------------------------------------------------------------

            relevance_penalty = (
                legal_relevance_penalty(
                    chunk,
                    query_info,
                )
            )

            score += relevance_penalty

            # -------------------------------------------------------------
            # Section mentioned in text
            # -------------------------------------------------------------

            section_text_score = 0.0

            if section_number:

                section_pattern = re.compile(
                    rf"\bsection\s+"
                    rf"{re.escape(section_number)}\b",
                    re.IGNORECASE,
                )

                if section_pattern.search(
                    chunk.get(
                        "text",
                        "",
                    )
                ):

                    section_text_score = 0.10

                    score += (
                        section_text_score
                    )

            # -------------------------------------------------------------
            # Diagnostics
            # -------------------------------------------------------------

            diagnostics[index] = {

                "exact_section": exact_section,

                "exact_act": exact_act,

                "exact_act_section": (
                    exact_act_section
                ),

                "concept_score": (
                    concept_score
                ),

                "title_score": (
                    title_score
                ),

                "intent_score": (
                    intent_score
                ),

                "act_score": (
                    act_score
                ),

                "section_text_score": (
                    section_text_score
                ),

                "relevance_penalty": (
                    relevance_penalty
                ),
            }

            final_scores[index] = score

        # =====================================================================
        # SORT
        # =====================================================================

        ranked_indices = sorted(
            final_scores,
            key=final_scores.get,
            reverse=True,
        )

        # =====================================================================
        # DEDUPLICATION
        # =====================================================================

        results = []

        seen_sections = set()

        for index in ranked_indices:

            chunk = self.chunks[index]

            section_number_chunk = (
                normalize_section_number(
                    chunk.get(
                        "section_number"
                    )
                )
            )

            document_id = (
                normalize_document_name(
                    chunk.get(
                        "document_id",
                        "",
                    )
                )
            )

            subsection = (
                chunk.get(
                    "subsection"
                )
            )

            # -----------------------------------------------------------------
            # For exact section queries, avoid showing multiple versions of
            # the same Act + section.
            # -----------------------------------------------------------------

            dedupe_key = (
                document_id,
                section_number_chunk,
                subsection,
            )

            if dedupe_key in seen_sections:

                continue

            seen_sections.add(
                dedupe_key
            )

            diagnostic = diagnostics[
                index
            ]

            result = chunk.copy()

            result.update({

                "rank": (
                    len(results)
                    + 1
                ),

                "fusion_score": float(
                    fused_scores.get(
                        index,
                        0.0,
                    )
                ),

                "final_score": float(
                    final_scores[index]
                ),

                "sparse_score": float(
                    sparse_scores[index]
                ),

                "dense_score": float(
                    dense_scores[index]
                ),

                "exact_section_match": (
                    diagnostic[
                        "exact_section"
                    ]
                ),

                "exact_act_match": (
                    diagnostic[
                        "exact_act"
                    ]
                ),

                "exact_act_section_match": (
                    diagnostic[
                        "exact_act_section"
                    ]
                ),

                "concept_score": float(
                    diagnostic[
                        "concept_score"
                    ]
                ),

                "section_title_score": float(
                    diagnostic[
                        "title_score"
                    ]
                ),

                "intent_score": float(
                    diagnostic[
                        "intent_score"
                    ]
                ),

                "act_score": float(
                    diagnostic[
                        "act_score"
                    ]
                ),

                "relevance_penalty": float(
                    diagnostic[
                        "relevance_penalty"
                    ]
                ),
            })

            results.append(
                result
            )

            if len(results) >= top_k:

                break

        return results


# =============================================================================
# DISPLAY QUERY ANALYSIS
# =============================================================================

def print_query_analysis(
    query: str,
):

    info = analyze_query(
        query
    )

    print(
        "\n"
        + "-"
        * 90
    )

    print(
        "QUERY ANALYSIS"
    )

    print(
        f"Original query : "
        f"{info['original_query']}"
    )

    print(
        f"Section number : "
        f"{info['section_number']}"
    )

    print(
        f"Act            : "
        f"{info['act']}"
    )

    print(
        f"Intent         : "
        f"{info['intent']}"
    )

    print(
        f"Section query  : "
        f"{info['is_section_query']}"
    )

    print(
        f"Concepts       : "
        f"{', '.join(info['concepts'])}"
    )

    print(
        f"Fuzzy matches  : "
        f"{', '.join(info['fuzzy_matches'])}"
    )

    print(
        "-"
        * 90
    )


# =============================================================================
# DISPLAY RESULTS
# =============================================================================

def print_results(
    results: list[dict],
):

    if not results:

        print(
            "\nNo results found."
        )

        return

    for result in results:

        print(
            "\n"
            + "="
            * 90
        )

        print(
            f"Rank: "
            f"{result['rank']}"
        )

        print(
            f"Document: "
            f"{result['document']}"
        )

        print(
            f"Document ID: "
            f"{result['document_id']}"
        )

        print(
            f"Category: "
            f"{result['category']}"
        )

        print(
            f"Chapter: "
            f"{result.get('chapter')}"
        )

        print(
            f"Section: "
            f"{result.get('section')}"
        )

        print(
            f"Section Number: "
            f"{result.get('section_number')}"
        )

        print(
            f"Section Title: "
            f"{result.get('section_title')}"
        )

        print(
            f"Subsection: "
            f"{result.get('subsection')}"
        )

        print(
            f"Page: "
            f"{result.get('page_number')}"
        )

        print(
            f"Exact Section Match: "
            f"{result['exact_section_match']}"
        )

        print(
            f"Exact Act Match: "
            f"{result['exact_act_match']}"
        )

        print(
            f"Exact Act + Section: "
            f"{result['exact_act_section_match']}"
        )

        print(
            f"Concept Score: "
            f"{result['concept_score']:.4f}"
        )

        print(
            f"Section Title Score: "
            f"{result['section_title_score']:.4f}"
        )

        print(
            f"Intent Score: "
            f"{result['intent_score']:.4f}"
        )

        print(
            f"Act Score: "
            f"{result['act_score']:.4f}"
        )

        print(
            f"Relevance Penalty: "
            f"{result['relevance_penalty']:.4f}"
        )

        print(
            f"Fusion Score: "
            f"{result['fusion_score']:.6f}"
        )

        print(
            f"Final Score: "
            f"{result['final_score']:.6f}"
        )

        print(
            f"BM25 Score: "
            f"{result['sparse_score']:.6f}"
        )

        print(
            f"Dense Score: "
            f"{result['dense_score']:.6f}"
        )

        print(
            "\nTEXT:"
        )

        print(
            result["text"][:2500]
        )


# =============================================================================
# BUILD RETRIEVER
# =============================================================================

def build_retriever() -> HybridRetriever:

    pdf_files = discover_pdfs()

    if not pdf_files:

        raise FileNotFoundError(
            f"No PDF files found in "
            f"{DATA_DIR}"
        )

    print(
        f"Project directory: "
        f"{PROJECT_DIR}"
    )

    print(
        f"Data directory: "
        f"{DATA_DIR}"
    )

    print(
        f"PDF files found: "
        f"{len(pdf_files)}"
    )

    print(
        "\nProcessing legal documents..."
    )

    chunks = process_documents(
        pdf_files
    )

    print(
        f"\nTotal chunks: "
        f"{len(chunks)}"
    )

    return HybridRetriever(
        chunks=chunks
    )


# =============================================================================
# MAIN
# =============================================================================

def main():

    retriever = build_retriever()

    print(
        "\n"
        + "="
        * 90
    )

    print(
        "LEGAL RAG RETRIEVER"
    )

    print(
        "="
        * 90
    )

    print(
        "\nType 'exit' to quit."
    )

    while True:

        query = input(
            "\nLegal query, or 'exit': "
        ).strip()

        if query.lower() == "exit":

            break

        if not query:

            continue

        # =====================================================================
        # QUERY ANALYSIS
        # =====================================================================

        print_query_analysis(
            query
        )

        # =====================================================================
        # SEARCH
        # =====================================================================

        results = retriever.search(
            query=query,
            top_k=DEFAULT_TOP_K,
            candidate_k=DEFAULT_CANDIDATE_K,
        )

        # =====================================================================
        # RESULTS
        # =====================================================================

        print_results(
            results
        )


# =============================================================================
# ENTRY POINT
# =============================================================================

if __name__ == "__main__":

    main()


Project directory: /home/bharadwaj/Workspace/india-legal-ai
Data directory: /home/bharadwaj/Workspace/india-legal-ai/backend/app/data
PDF files found: 10

Processing legal documents...
contract_act.pdf: 53 pages -> 588 chunks
transfer_of_property_act.pdf: 46 pages -> 442 chunks
constitution_of_india.pdf: 268 pages -> 401 chunks
bns_2023.pdf: 102 pages -> 363 chunks
bnss_2023.pdf: 249 pages -> 546 chunks
bsa_2023.pdf: 47 pages -> 175 chunks
crpc_1973.pdf: 368 pages -> 1109 chunks
evidence_act_1872.pdf: 60 pages -> 494 chunks
ipc_1860.pdf: 258 pages -> 624 chunks
information_technology_act.pdf: 36 pages -> 333 chunks

Total chunks: 5075
Building BM25 index...
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Creating embeddings...


Batches:   0%|          | 0/159 [00:00<?, ?it/s]

Indexed 691 unique section numbers.
Indexed 3243 Act + section combinations.

LEGAL RAG RETRIEVER

Type 'exit' to quit.



Legal query, or 'exit':  What is Section IPC 420?



------------------------------------------------------------------------------------------
QUERY ANALYSIS
Original query : What is Section IPC 420?
Section number : 420
Act            : ipc_1860
Intent         : definition
Section query  : True
Concepts       : 
Fuzzy matches  : 
------------------------------------------------------------------------------------------

Rank: 1
Document: ipc_1860
Document ID: ipc_1860
Category: criminal
Chapter: CHAPTER XVII - OF OFFENCES AGAINST PROPERTY
Section: Section 420
Section Number: 420
Section Title: Cheating and dishonestly inducing delivery of property —
Subsection: None
Page: 216
Exact Section Match: True
Exact Act Match: True
Exact Act + Section: True
Concept Score: 0.0000
Section Title Score: 0.0000
Intent Score: 0.0500
Act Score: 2.5000
Relevance Penalty: 0.0000
Fusion Score: 0.014286
Final Score: 6.064286
BM25 Score: 7.127297
Dense Score: 0.173246

TEXT:
420. Cheating and dishonestly inducing delivery of property —
Whoever cheats and 